## **Import & Setup**

In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

## **Load the Interaction Matrix**

In [2]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

DATA_DIR = Path('../data')
matrix = pd.read_parquet(DATA_DIR / 'interaction_matrix_raw.parquet')
print(f'Matrix shape: {matrix.shape}')

Matrix shape: (306, 48)


## **Compute eligibility distribution**

In order to apply the LOO (Leave-One-Out) training approach, we need to have at least 2 positive instances per neighborhood (1 for positive training and one for testing)

In [3]:
# Number of non-zero neighbourhoods per NACE class
nz_per_nace = (matrix > 0).sum(axis=1)

print('Non-zero neighbourhoods per NACE class:')
print(nz_per_nace.describe())
print()
print(f'Classes with 1 non-zero neighbourhood (train-only, NOT testable): {(nz_per_nace == 1).sum()}')
print(f'Classes with 2 non-zero (test, no val):                            {(nz_per_nace == 2).sum()}')
print(f'Classes with >=3 non-zero (test + val):                            {(nz_per_nace >= 3).sum()}')
print()
print(f'Total testable classes (>=2 non-zero): {(nz_per_nace >= 2).sum()}')

Non-zero neighbourhoods per NACE class:
count    306.000000
mean       5.562092
std        4.785675
min        1.000000
25%        2.000000
50%        4.000000
75%        8.000000
max       25.000000
dtype: float64

Classes with 1 non-zero neighbourhood (train-only, NOT testable): 62
Classes with 2 non-zero (test, no val):                            44
Classes with >=3 non-zero (test + val):                            200

Total testable classes (>=2 non-zero): 244


## **Perform the Split**

In [4]:
train_matrix = matrix.copy()
val_records = []   # (nace, neighborhood, original_count)
test_records = []

for nace in matrix.index:
    nonzero_nbhds = matrix.columns[matrix.loc[nace] > 0].tolist()
    n_nz = len(nonzero_nbhds)

    if n_nz >= 3:
        # hold out one for test, one for validation
        chosen = rng.choice(nonzero_nbhds, size=2, replace=False)
        test_nbhd, val_nbhd = chosen[0], chosen[1]
        test_records.append((nace, test_nbhd, int(matrix.loc[nace, test_nbhd])))
        val_records.append((nace, val_nbhd, int(matrix.loc[nace, val_nbhd])))
        train_matrix.loc[nace, test_nbhd] = 0
        train_matrix.loc[nace, val_nbhd] = 0
    elif n_nz == 2:
        # test only, no validation
        test_nbhd = rng.choice(nonzero_nbhds)
        test_records.append((nace, test_nbhd, int(matrix.loc[nace, test_nbhd])))
        train_matrix.loc[nace, test_nbhd] = 0
    # n_nz == 1: train-only, no holdout

test_df = pd.DataFrame(test_records, columns=['nace', 'neighborhood', 'original_count'])
val_df = pd.DataFrame(val_records, columns=['nace', 'neighborhood', 'original_count'])

print(f'Train interactions remaining: {int((train_matrix > 0).sum().sum())}')
print(f'Validation instances: {len(val_df)}')
print(f'Test instances:       {len(test_df)}')

Train interactions remaining: 1258
Validation instances: 200
Test instances:       244


## **Validate the Split**

In [8]:
# 1. No leakage: test/val cells must be zero in train_matrix
for _, r in test_df.iterrows():
    assert train_matrix.loc[r['nace'], r['neighborhood']] == 0, 'Test leak!'
for _, r in val_df.iterrows():
    assert train_matrix.loc[r['nace'], r['neighborhood']] == 0, 'Val leak!'

# 2. Every testable NACE class still has at least one training interaction
testable_nace = pd.concat([test_df['nace'], val_df['nace']]).unique()
for nace in testable_nace:
    assert (train_matrix.loc[nace] > 0).sum() >= 1, f'{nace} has no training signal!'

# 3. Accounting: train + val + test interactions == original non-zero cells
original_nz = int((matrix > 0).sum().sum())
split_total = int((train_matrix > 0).sum().sum()) + len(val_df) + len(test_df)
assert original_nz == split_total, f'Accounting mismatch: {original_nz} vs {split_total}'

print('All split validation checks passed.')
print(f'Original non-zero cells: {original_nz}')
print(f'  Train: {int((train_matrix > 0).sum().sum())}')
print(f'  Val:   {len(val_df)}')
print(f'  Test:  {len(test_df)}')

All split validation checks passed.
Original non-zero cells: 1702
  Train: 1258
  Val:   200
  Test:  244


## **Save**

In [ ]:
train_matrix.to_parquet(DATA_DIR / 'train_matrix.parquet')
test_df.to_parquet(DATA_DIR / 'test_pairs.parquet')
val_df.to_parquet(DATA_DIR / 'val_pairs.parquet')

split_metadata = {
    'random_seed': RANDOM_SEED,
    'protocol': 'leave-one-out (test); leave-one-out (val) for classes with >=3 non-zero neighborhoods',
    'n_train_interactions': int((train_matrix > 0).sum().sum()),
    'n_val': len(val_df),
    'n_test': len(test_df),
    'testable_nace_classes': len(test_df),
}
with open(DATA_DIR / 'split_metadata.json', 'w') as f:
    json.dump(split_metadata, f, indent=2)

print('Saved train_matrix, test_pairs, val_pairs, split_metadata.')

Saved train_matrix, test_pairs, val_pairs, split_metadata.


: 